In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
!pip install -U langchain langchain-text-splitters sentence-transformers faiss-cpu

In [ ]:
#Orignal Paper Chunking Format (Chunk per character + Sliding Windows)
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# Load the txt file
txt_path = "Irish_RAG_Pairs.txt"

with open(txt_path, "r", encoding="utf-8") as file:
    text = file.read()



# Step 2: Chunk the text
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
original_chunks = text_splitter.split_text(text)

print(f"Total Chunks Created: {len(original_chunks)}")  # Number of chunks

# Step 3: Convert chunks into embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
original_embeddings = model.encode(original_chunks)

dimension = original_embeddings.shape[1]
original_index = faiss.IndexFlatL2(dimension)
original_index.add(original_embeddings)

print(f"Total Original Chunks Stored in FAISS: {original_index.ntotal}")
print(f"Embedding Dimension: {dimension}")


The Irish RAG pairs file was successfully chunked into 198 chunks. Embeddings were generated using all-MiniLM-L6-v2 and stored in a FAISS index with embedding dimension 384, matching the original English chunking pipeline.

In [ ]:
#Sentence Pair Chunking Format (Chunk per sentence pair)

#Load the txt file
txt_path = "Irish_RAG_Pairs.txt"

with open(txt_path, "r", encoding="utf-8") as file:
    text = file.read()


#Chunk the text by sentence pair
raw_chunks = text.split("\n\n")

sentence_pair_chunks = []

for chunk in raw_chunks:
    chunk = chunk.strip()

    if (
        "Original Sentence:" in chunk
        and "Counterfactual Sentence:" in chunk
    ):
        sentence_pair_chunks.append(chunk)

print(f"Total Sentence Pair Chunks Created: {len(sentence_pair_chunks)}")


#Convert chunks into embeddings
model = SentenceTransformer("all-MiniLM-L6-v2")
sentence_pair_embeddings = model.encode(sentence_pair_chunks)

dimension = sentence_pair_embeddings.shape[1]
sentence_pair_index = faiss.IndexFlatL2(dimension)
sentence_pair_index.add(sentence_pair_embeddings)

print(
    f"Total Sentence Pair Chunks Stored in FAISS: "
    f"{sentence_pair_index.ntotal}")